# ADS Homework 3 - Part 1: MLP (PyTorch)


This notebook follows `.cursor/rules/task_description_hw3.mdc` and the plan in `hw3_plan.md`.


**Dataset (Kaggle path):**
- **Telco Customer Churn**: `/kaggle/input/telco-customer-churn-realistic-customer-feedback/telco_churn_with_all_feedback.csv`

We use PyTorch for MLP experiments with systematic architecture and optimization changes.

**Author:** [Your Name]


In [ ]:
# Core imports and setup
import os
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Subset

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


## 1) Dataset Path
Confirm this path matches the Kaggle mount.

- **Telco Customer Churn**: `/kaggle/input/telco-customer-churn-realistic-customer-feedback/telco_churn_with_all_feedback.csv`


In [ ]:
# Update if your Kaggle path differs
TELCO_PATH = Path("/kaggle/input/telco-customer-churn-realistic-customer-feedback/telco_churn_with_all_feedback.csv")

print("Telco Customer Churn exists:", TELCO_PATH.exists())


# Part 1: MLP on Telco Customer Churn (Tabular)

**Tasks:**
1. Binary classification: predict `Churn` (Yes/No -> 1/0)
2. Regression: predict `TotalCharges` (numeric; impute median)

**Preprocessing:**
- Numeric: median imputation + standard scaling
- Categorical: most-frequent imputation + one-hot encoding

**Show:**
- Training and validation performance
- Loss curves
- Final evaluation metrics (classification + regression)

**Experiments (discuss effects with short comments):**
- **Training & Optimization:** optimizers (SGD, SGD+momentum, Adam), learning rate too small/good/too large, learning rate scheduling, batch size, early stopping, number of epochs
- **Architecture & Representation:** depth, width, activations (ReLU, LeakyReLU, Tanh, Sigmoid), weight initialization (Xavier, He, random), batch normalization
- **Regularization & Stability:** L1/L2 weight regularization, activity regularization, dropout, gradient clipping (optional)


In [ ]:
def build_preprocessor(df, target_cols):
    feature_cols = [c for c in df.columns if c not in target_cols]
    X = df[feature_cols]

    numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
        ]
    )

    return preprocessor

if TELCO_PATH.exists():
    df_telco = pd.read_csv(TELCO_PATH)

    churn_col = next((c for c in df_telco.columns if c.lower() == "churn"), None)
    if churn_col is None:
        raise ValueError("Churn column not found.")

    target_reg = "TotalCharges"
    if target_reg not in df_telco.columns:
        raise ValueError("TotalCharges column not found.")

    df_telco[target_reg] = pd.to_numeric(df_telco[target_reg], errors="coerce")
    median_total = df_telco[target_reg].median()
    df_telco[target_reg] = df_telco[target_reg].fillna(median_total)

    y_bin = df_telco[churn_col].apply(
        lambda x: 1 if str(x).lower() in ["yes", "true", "1"] else 0
    )

    id_cols = [c for c in df_telco.columns if c.lower() in ["customerid", "customer_id"]]

    feature_df = df_telco.drop(columns=[churn_col] + id_cols)
    preprocessor = build_preprocessor(feature_df, target_cols=[target_reg])

    X_cls = feature_df.drop(columns=[target_reg])

    X_all = preprocessor.fit_transform(X_cls)
    X_all = np.nan_to_num(X_all, nan=0.0, posinf=0.0, neginf=0.0)

    y_reg = df_telco[target_reg].values.astype(np.float32)
    y_reg = np.nan_to_num(y_reg, nan=np.nanmedian(y_reg), posinf=0.0, neginf=0.0)
    y_bin = y_bin.values.astype(np.float32)

    print("Feature matrix shape:", X_all.shape)

    X_train, X_val, y_bin_train, y_bin_val, y_reg_train, y_reg_val = train_test_split(
        X_all, y_bin, y_reg, test_size=0.2, random_state=SEED, stratify=y_bin
    )

    train_ds_cls = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_bin_train, dtype=torch.float32),
    )
    val_ds_cls = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_bin_val, dtype=torch.float32),
    )

    train_ds_reg = TensorDataset(
        torch.tensor(X_train, dtype=torch.float32),
        torch.tensor(y_reg_train, dtype=torch.float32),
    )
    val_ds_reg = TensorDataset(
        torch.tensor(X_val, dtype=torch.float32),
        torch.tensor(y_reg_val, dtype=torch.float32),
    )
else:
    print("Warning: Telco dataset not found. Skipping Part 1 data loading.")


In [ ]:
import copy

class MLP(nn.Module):
    def __init__(
        self,
        input_dim,
        hidden_sizes=(128, 64),
        dropout=0.2,
        activation="relu",
        use_batch_norm=False,
        out_dim=1,
        init="auto",
    ):
        super().__init__()
        layers = []
        prev = input_dim

        if activation == "relu":
            Act = nn.ReLU
        elif activation == "leakyrelu":
            Act = nn.LeakyReLU
        elif activation == "tanh":
            Act = nn.Tanh
        elif activation == "sigmoid":
            Act = nn.Sigmoid
        else:
            Act = nn.ReLU

        for h in hidden_sizes:
            layers.append(nn.Linear(prev, h))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(Act())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
            prev = h

        layers.append(nn.Linear(prev, out_dim))
        self.layers = nn.ModuleList(layers)

        self._init_weights(init, activation)

    def _init_weights(self, init, activation):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                if init == "auto":
                    if activation in ["relu", "leakyrelu"]:
                        nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                    else:
                        nn.init.xavier_normal_(m.weight)
                elif init == "he":
                    nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                elif init == "xavier":
                    nn.init.xavier_normal_(m.weight)
                elif init == "random":
                    nn.init.normal_(m.weight, mean=0.0, std=0.01)
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)

    def forward(self, x, return_activations=False):
        activations = []
        for layer in self.layers:
            x = layer(x)
            if return_activations and isinstance(
                layer, (nn.ReLU, nn.LeakyReLU, nn.Tanh, nn.Sigmoid)
            ):
                activations.append(x)
        if return_activations:
            return x, activations
        return x

class EarlyStopping:
    def __init__(self, patience=5, min_delta=0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = None
        self.counter = 0
        self.best_state = None

    def step(self, val_loss, model):
        if self.best_loss is None or val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_state = copy.deepcopy(model.state_dict())
            return False
        self.counter += 1
        return self.counter >= self.patience

def train_mlp(
    model,
    train_ds,
    val_ds,
    loss_fn,
    optimizer,
    scheduler=None,
    epochs=20,
    batch_size=256,
    early_stopping=None,
    l1_lambda=0.0,
    activity_lambda=0.0,
    clip_grad=None,
):
    model.to(DEVICE)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)

    history = {"train_loss": [], "val_loss": []}
    stopper = EarlyStopping(**early_stopping) if early_stopping else None

    start_time = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        train_losses = []
        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()

            if activity_lambda > 0:
                preds, activations = model(xb, return_activations=True)
            else:
                preds = model(xb)
                activations = []

            preds = preds.squeeze()
            loss = loss_fn(preds, yb)

            if l1_lambda > 0:
                l1_penalty = sum(p.abs().sum() for p in model.parameters())
                loss = loss + l1_lambda * l1_penalty

            if activity_lambda > 0 and activations:
                activity_penalty = torch.stack([a.pow(2).mean() for a in activations]).mean()
                loss = loss + activity_lambda * activity_penalty

            loss.backward()
            if clip_grad is not None:
                torch.nn.utils.clip_grad_norm_(model.parameters(), clip_grad)
            optimizer.step()
            train_losses.append(loss.item())

        model.eval()
        val_losses = []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                preds = model(xb).squeeze()
                loss = loss_fn(preds, yb)
                val_losses.append(loss.item())

        history["train_loss"].append(np.mean(train_losses))
        history["val_loss"].append(np.mean(val_losses))

        if scheduler is not None:
            if isinstance(scheduler, optim.lr_scheduler.ReduceLROnPlateau):
                scheduler.step(history["val_loss"][-1])
            else:
                scheduler.step()

        if epoch % 5 == 0 or epoch == 1:
            print(
                f"Epoch {epoch:02d} | Train Loss {history['train_loss'][-1]:.4f} | "
                f"Val Loss {history['val_loss'][-1]:.4f}"
            )

        if stopper and stopper.step(history["val_loss"][-1], model):
            print(f"Early stopping at epoch {epoch}")
            break

    print(f"Training finished in {time.time() - start_time:.2f}s")
    return history, (stopper.best_state if stopper else None)


def eval_classification(model, X, y_true):
    model.eval()
    with torch.no_grad():
        logits = model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).squeeze().cpu().numpy()
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= 0.5).astype(int)
    return {
        "accuracy": accuracy_score(y_true, preds),
        "f1": f1_score(y_true, preds),
        "roc_auc": roc_auc_score(y_true, probs),
    }


def eval_regression(model, X, y_true):
    model.eval()
    with torch.no_grad():
        preds = model(torch.tensor(X, dtype=torch.float32).to(DEVICE)).squeeze().cpu().numpy()
    return {
        "mae": mean_absolute_error(y_true, preds),
        "mse": mean_squared_error(y_true, preds),
        "r2": r2_score(y_true, preds),
    }


def plot_loss_curves(history, title):
    plt.figure(figsize=(8, 4))
    plt.plot(history["train_loss"], label="train")
    plt.plot(history["val_loss"], label="val")
    plt.title(title)
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()


def print_metrics(label, metrics):
    metrics_str = ", ".join([f"{k}={v:.4f}" for k, v in metrics.items()])
    print(f"{label}: {metrics_str}")


### MLP Experiments (based on the plan)
We will run targeted experiments in three groups. To keep the notebook readable, only a subset of loss curves is plotted.

**Training & Optimization**
- Optimizers: SGD, SGD+momentum, Adam
- Learning rate: too small / good / too large
- Learning rate scheduling
- Batch size
- Early stopping
- Number of epochs

**Architecture & Representation**
- Depth (hidden layers)
- Width (neurons per layer)
- Activations: ReLU, LeakyReLU, Tanh, Sigmoid
- Weight initialization: Xavier, He, random
- Batch Normalization

**Regularization & Stability**
- L1 / L2 weight regularization
- Activity regularization
- Dropout
- Gradient clipping (optional)


In [ ]:
if TELCO_PATH.exists():
    RUN_MLP_EXPERIMENTS = True
    RUN_REGRESSION_EXPERIMENTS = True
    MLP_EXPERIMENT_LIMIT = None  # Set to an int for a quicker run
    PLOT_LOSS_CURVES = True
    PLOT_MAX = 8

    BASE_CONFIG = {
        "hidden_sizes": (128, 64),
        "dropout": 0.2,
        "activation": "relu",
        "use_batch_norm": False,
        "init": "he",
        "optimizer": "adam",
        "lr": 1e-3,
        "momentum": 0.9,
        "batch_size": 256,
        "epochs": 15,
        "weight_decay": 0.0,
        "l1_lambda": 0.0,
        "activity_lambda": 0.0,
        "clip_grad": None,
        "scheduler": None,
        "scheduler_params": {},
        "early_stopping": None,
    }

    MLP_EXPERIMENTS = [
        {"name": "baseline_adam", "group": "training_opt"},
        {"name": "optimizer_sgd", "group": "training_opt", "optimizer": "sgd", "lr": 0.01, "momentum": 0.0},
        {"name": "optimizer_sgd_momentum", "group": "training_opt", "optimizer": "sgd", "lr": 0.01, "momentum": 0.9},
        {"name": "lr_too_small", "group": "training_opt", "lr": 1e-5},
        {"name": "lr_too_large", "group": "training_opt", "lr": 1e-1},
        {
            "name": "lr_scheduler_step",
            "group": "training_opt",
            "scheduler": "step",
            "scheduler_params": {"step_size": 5, "gamma": 0.5},
        },
        {"name": "batch_size_small", "group": "training_opt", "batch_size": 64},
        {"name": "batch_size_large", "group": "training_opt", "batch_size": 512},
        {
            "name": "early_stopping",
            "group": "training_opt",
            "epochs": 40,
            "early_stopping": {"patience": 4, "min_delta": 1e-3},
        },
        {"name": "epochs_short", "group": "training_opt", "epochs": 5},
        {"name": "epochs_long", "group": "training_opt", "epochs": 25},
        {"name": "depth_shallow", "group": "architecture", "hidden_sizes": (64,)},
        {"name": "depth_deep", "group": "architecture", "hidden_sizes": (256, 128, 64)},
        {"name": "width_narrow", "group": "architecture", "hidden_sizes": (64, 32)},
        {"name": "width_wide", "group": "architecture", "hidden_sizes": (256, 128)},
        {"name": "activation_leakyrelu", "group": "architecture", "activation": "leakyrelu"},
        {"name": "activation_tanh", "group": "architecture", "activation": "tanh"},
        {"name": "activation_sigmoid", "group": "architecture", "activation": "sigmoid"},
        {"name": "init_xavier", "group": "architecture", "init": "xavier"},
        {"name": "init_random", "group": "architecture", "init": "random"},
        {"name": "batch_norm", "group": "architecture", "use_batch_norm": True},
        {"name": "l2_weight_decay", "group": "regularization", "weight_decay": 1e-4},
        {"name": "l1_weight", "group": "regularization", "l1_lambda": 1e-6},
        {"name": "activity_reg", "group": "regularization", "activity_lambda": 1e-6},
        {"name": "dropout_high", "group": "regularization", "dropout": 0.5},
        {"name": "grad_clip", "group": "regularization", "clip_grad": 1.0},
    ]

    if MLP_EXPERIMENT_LIMIT is not None:
        MLP_EXPERIMENTS = MLP_EXPERIMENTS[:MLP_EXPERIMENT_LIMIT]

    def build_optimizer(params, cfg):
        if cfg["optimizer"] == "sgd":
            return optim.SGD(
                params,
                lr=cfg["lr"],
                momentum=cfg["momentum"],
                weight_decay=cfg["weight_decay"],
            )
        return optim.Adam(params, lr=cfg["lr"], weight_decay=cfg["weight_decay"])

    def build_scheduler(optimizer, cfg):
        if cfg["scheduler"] == "step":
            params = cfg.get("scheduler_params", {})
            return optim.lr_scheduler.StepLR(
                optimizer,
                step_size=params.get("step_size", 5),
                gamma=params.get("gamma", 0.5),
            )
        if cfg["scheduler"] == "plateau":
            params = cfg.get("scheduler_params", {})
            return optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="min",
                patience=params.get("patience", 2),
                factor=params.get("factor", 0.5),
            )
        return None

    def run_experiments(task_name, train_ds, val_ds, X_train, y_train, X_val, y_val, loss_fn):
        results = []
        plot_count = 0

        for cfg in MLP_EXPERIMENTS:
            exp_cfg = dict(BASE_CONFIG)
            exp_cfg.update(cfg)

            model = MLP(
                input_dim=X_train.shape[1],
                hidden_sizes=exp_cfg["hidden_sizes"],
                dropout=exp_cfg["dropout"],
                activation=exp_cfg["activation"],
                use_batch_norm=exp_cfg["use_batch_norm"],
                init=exp_cfg["init"],
            )
            optimizer = build_optimizer(model.parameters(), exp_cfg)
            scheduler = build_scheduler(optimizer, exp_cfg)

            print(f"\n[{task_name.upper()}] {exp_cfg['name']}")
            history, best_state = train_mlp(
                model,
                train_ds,
                val_ds,
                loss_fn,
                optimizer,
                scheduler=scheduler,
                epochs=exp_cfg["epochs"],
                batch_size=exp_cfg["batch_size"],
                early_stopping=exp_cfg["early_stopping"],
                l1_lambda=exp_cfg["l1_lambda"],
                activity_lambda=exp_cfg["activity_lambda"],
                clip_grad=exp_cfg["clip_grad"],
            )

            if best_state is not None:
                model.load_state_dict(best_state)

            if task_name == "classification":
                train_metrics = eval_classification(model, X_train, y_train)
                val_metrics = eval_classification(model, X_val, y_val)
            else:
                train_metrics = eval_regression(model, X_train, y_train)
                val_metrics = eval_regression(model, X_val, y_val)

            print_metrics("Train", train_metrics)
            print_metrics("Val", val_metrics)

            if PLOT_LOSS_CURVES and plot_count < PLOT_MAX:
                plot_loss_curves(history, f"{task_name} - {exp_cfg['name']}")
                plot_count += 1

            result_row = {
                "task": task_name,
                "group": exp_cfg["group"],
                "name": exp_cfg["name"],
                "optimizer": exp_cfg["optimizer"],
                "lr": exp_cfg["lr"],
                "batch_size": exp_cfg["batch_size"],
                "epochs": exp_cfg["epochs"],
                "hidden_sizes": str(exp_cfg["hidden_sizes"]),
                "activation": exp_cfg["activation"],
                "init": exp_cfg["init"],
                "use_batch_norm": exp_cfg["use_batch_norm"],
                "dropout": exp_cfg["dropout"],
                "weight_decay": exp_cfg["weight_decay"],
                "l1_lambda": exp_cfg["l1_lambda"],
                "activity_lambda": exp_cfg["activity_lambda"],
                "clip_grad": exp_cfg["clip_grad"],
                "scheduler": exp_cfg["scheduler"],
            }
            for k, v in train_metrics.items():
                result_row[f"train_{k}"] = v
            for k, v in val_metrics.items():
                result_row[f"val_{k}"] = v

            results.append(result_row)

        return pd.DataFrame(results)

    results_cls = None
    results_reg = None

    if RUN_MLP_EXPERIMENTS:
        results_cls = run_experiments(
            "classification",
            train_ds_cls,
            val_ds_cls,
            X_train,
            y_bin_train,
            X_val,
            y_bin_val,
            nn.BCEWithLogitsLoss(),
        )

    if RUN_REGRESSION_EXPERIMENTS:
        results_reg = run_experiments(
            "regression",
            train_ds_reg,
            val_ds_reg,
            X_train,
            y_reg_train,
            X_val,
            y_reg_val,
            nn.MSELoss(),
        )

    if results_cls is not None:
        print("\nClassification results (head):")
        print(results_cls.head())

    if results_reg is not None:
        print("\nRegression results (head):")
        print(results_reg.head())


### Discussion Question 1 (MLP)
* **Why are neural networks so powerful?**
* **Why does training become more difficult as we go deeper?**
* *(Optional) If MLPs can approximate any function with a single hidden layer, what unique benefits does depth provide beyond width?*

*(Double-click to edit and answer here)*


## Wrap-up
- Summarize key findings.
- Optional: short error analysis.
